# QA 3 (Unit 3): Naive Bayes — Manual + Code Implementation

**Author:** Student Submission  
**Topic:** Spam Classification using Naive Bayes  

---

## How Naive Bayes Works

Naive Bayes is a probabilistic classifier based on **Bayes' Theorem**:

$$P(C \mid X) = \frac{P(X \mid C) \cdot P(C)}{P(X)}$$

- **Prior** `P(C)`: Probability of a class before seeing any features (how often spam occurs in data).
- **Likelihood** `P(X|C)`: Probability of observing features given the class (how likely each word appears in spam).
- **Posterior** `P(C|X)`: Updated probability of class given observed features — what we want.

The **"naive"** assumption: all features (words) are **conditionally independent** given the class.

$$P(C \mid w_1, w_2, ..., w_n) \propto P(C) \cdot \prod_{i=1}^{n} P(w_i \mid C)$$


---
## Part 1: Manual Naive Bayes (Step-by-Step)

### Dataset (Training)

| Message | Label |
|---|---|
| free money win prize | Spam |
| win free lottery now | Spam |
| free prize offer win | Spam |
| meeting at noon today | Ham |
| lunch today at office | Ham |
| please call me today | Ham |

### Test Message: `"free money today"`

In [1]:
# ─── Manual Naive Bayes (Part 1) ───────────────────────────────────────────
import numpy as np
from collections import defaultdict

# Training data
training_data = [
    ("free money win prize",    "spam"),
    ("win free lottery now",    "spam"),
    ("free prize offer win",    "spam"),
    ("meeting at noon today",   "ham"),
    ("lunch today at office",   "ham"),
    ("please call me today",    "ham"),
]

# ── Step 1: Count documents per class ──────────────────────────────────────
class_counts = defaultdict(int)
for _, label in training_data:
    class_counts[label] += 1

total_docs = len(training_data)
classes = list(class_counts.keys())

print("=" * 55)
print("STEP 1: Document Counts")
print("=" * 55)
for cls in classes:
    print(f"  {cls.upper():5s}: {class_counts[cls]} documents")
print(f"  TOTAL: {total_docs} documents")

STEP 1: Document Counts
  SPAM : 3 documents
  HAM  : 3 documents
  TOTAL: 6 documents


In [2]:
# ── Step 2: Compute Priors ─────────────────────────────────────────────────
priors = {cls: class_counts[cls] / total_docs for cls in classes}

print("=" * 55)
print("STEP 2: Prior Probabilities  P(Class)")
print("=" * 55)
for cls in classes:
    print(f"  P({cls.upper():5s}) = {class_counts[cls]}/{total_docs} = {priors[cls]:.4f}")

STEP 2: Prior Probabilities  P(Class)
  P(SPAM ) = 3/6 = 0.5000
  P(HAM  ) = 3/6 = 0.5000


In [3]:
# ── Step 3: Build Word Frequency Table ────────────────────────────────────
word_counts  = {cls: defaultdict(int) for cls in classes}
class_totals = defaultdict(int)
vocabulary   = set()

for message, label in training_data:
    words = message.lower().split()
    for word in words:
        word_counts[label][word] += 1
        class_totals[label]      += 1
        vocabulary.add(word)

vocab_size = len(vocabulary)
print("=" * 55)
print("STEP 3: Word Frequencies per Class")
print("=" * 55)
print(f"  Vocabulary size (V) = {vocab_size}")
all_words = sorted(vocabulary)
header = f"  {'Word':<12}" + "".join(f"  {c.upper():>6}" for c in classes)
print(header)
print("  " + "-"*40)
for word in all_words:
    row = f"  {word:<12}" + "".join(f"  {word_counts[c][word]:>6}" for c in classes)
    print(row)
print("  " + "-"*40)
totals_row = f"  {'TOTAL':<12}" + "".join(f"  {class_totals[c]:>6}" for c in classes)
print(totals_row)

STEP 3: Word Frequencies per Class
  Vocabulary size (V) = 16
  Word            SPAM     HAM
  ----------------------------------------
  at                 0       2
  call               0       1
  free               3       0
  lottery            1       0
  lunch              0       1
  me                 0       1
  meeting            0       1
  money              1       0
  noon               0       1
  now                1       0
  offer              1       0
  office             0       1
  please             0       1
  prize              2       0
  today              0       3
  win                3       0
  ----------------------------------------
  TOTAL             12      12


In [4]:
# ── Step 4: Compute Likelihoods with Laplace Smoothing ────────────────────
# P(word | class) = (count(word, class) + 1) / (total_words_in_class + V)

def likelihood(word, cls):
    return (word_counts[cls][word] + 1) / (class_totals[cls] + vocab_size)

test_message = "free money today"
test_words   = test_message.lower().split()

print("=" * 55)
print("STEP 4: Likelihoods  P(word | Class)  [Laplace +1]")
print("=" * 55)
print(f"  Formula: P(w|C) = (count(w,C) + 1) / (total_C + V)")
print()
for word in test_words:
    for cls in classes:
        c = word_counts[cls][word]
        t = class_totals[cls]
        p = likelihood(word, cls)
        print(f"  P('{word}' | {cls.upper():5s}) = ({c}+1)/({t}+{vocab_size}) = {c+1}/{t+vocab_size} = {p:.4f}")
    print()

STEP 4: Likelihoods  P(word | Class)  [Laplace +1]
  Formula: P(w|C) = (count(w,C) + 1) / (total_C + V)

  P('free' | SPAM ) = (3+1)/(12+16) = 4/28 = 0.1429
  P('free' | HAM  ) = (0+1)/(12+16) = 1/28 = 0.0357

  P('money' | SPAM ) = (1+1)/(12+16) = 2/28 = 0.0714
  P('money' | HAM  ) = (0+1)/(12+16) = 1/28 = 0.0357

  P('today' | SPAM ) = (0+1)/(12+16) = 1/28 = 0.0357
  P('today' | HAM  ) = (3+1)/(12+16) = 4/28 = 0.1429



In [5]:
# ── Step 5: Compute Posterior & Classify ──────────────────────────────────
print("=" * 55)
print("STEP 5: Posterior Score  P(C) × ∏ P(wi|C)")
print("=" * 55)
print(f"  Test message: '{test_message}'")
print()

scores     = {}
log_scores = {}

for cls in classes:
    prior  = priors[cls]
    score  = prior
    log_sc = np.log(prior)
    parts  = [f"P({cls.upper()})={prior:.4f}"]

    for word in test_words:
        lh     = likelihood(word, cls)
        score  *= lh
        log_sc += np.log(lh)
        parts.append(f"P('{word}'|{cls.upper()})={lh:.4f}")

    scores[cls]     = score
    log_scores[cls] = log_sc
    print(f"  [{cls.upper()}]")
    print("  " + " × ".join(parts))
    print(f"  Score        = {score:.8f}")
    print(f"  Log Score    = {log_sc:.4f}")
    print()

# Normalize to get true probabilities
total_score = sum(scores.values())
print("-" * 55)
print("  Normalized Posterior Probabilities:")
for cls in classes:
    pct = scores[cls] / total_score * 100
    print(f"  P({cls.upper():5s} | message) ≈ {pct:.2f}%")

prediction = max(scores, key=scores.get)
print()
print(f"  ✅ MANUAL PREDICTION: {prediction.upper()}")

STEP 5: Posterior Score  P(C) × ∏ P(wi|C)
  Test message: 'free money today'

  [SPAM]
  P(SPAM)=0.5000 × P('free'|SPAM)=0.1429 × P('money'|SPAM)=0.0714 × P('today'|SPAM)=0.0357
  Score        = 0.00018222
  Log Score    = -8.6103

  [HAM]
  P(HAM)=0.5000 × P('free'|HAM)=0.0357 × P('money'|HAM)=0.0357 × P('today'|HAM)=0.1429
  Score        = 0.00009111
  Log Score    = -9.3035

-------------------------------------------------------
  Normalized Posterior Probabilities:
  P(SPAM  | message) ≈ 66.67%
  P(HAM   | message) ≈ 33.33%

  ✅ MANUAL PREDICTION: SPAM


---
## Part 2: Naive Bayes with sklearn

We now implement the same classification using **scikit-learn's `MultinomialNB`** and `CountVectorizer`, then compare results with the manual calculation.

In [6]:
# ─── sklearn Implementation (Part 2) ───────────────────────────────────────
from sklearn.naive_bayes      import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Same training data
messages = [msg for msg, _ in training_data]
labels   = [lbl for _, lbl in training_data]

# Vectorize
vectorizer = CountVectorizer()
X_train    = vectorizer.fit_transform(messages)
y_train    = labels

# Show vocabulary mapping
print("=" * 55)
print("CountVectorizer Vocabulary (word → index)")
print("=" * 55)
vocab_df = pd.DataFrame(list(vectorizer.vocabulary_.items()),
                        columns=["Word", "Index"]).sort_values("Index")
print(vocab_df.to_string(index=False))

print()
print("=" * 55)
print("Feature Matrix (each row = one training message)")
print("=" * 55)
feat_df = pd.DataFrame(X_train.toarray(),
                       columns=vectorizer.get_feature_names_out(),
                       index=[f"{lbl}: {msg[:25]}" for msg, lbl in training_data])
print(feat_df)

CountVectorizer Vocabulary (word → index)
   Word  Index
     at      0
   call      1
   free      2
lottery      3
  lunch      4
     me      5
meeting      6
  money      7
   noon      8
    now      9
  offer     10
 office     11
 please     12
  prize     13
  today     14
    win     15

Feature Matrix (each row = one training message)
                            at  call  free  lottery  lunch  me  meeting  \
spam: free money win prize   0     0     1        0      0   0        0   
spam: win free lottery now   0     0     1        1      0   0        0   
spam: free prize offer win   0     0     1        0      0   0        0   
ham: meeting at noon today   1     0     0        0      0   0        1   
ham: lunch today at office   1     0     0        0      1   0        0   
ham: please call me today    0     1     0        0      0   1        0   

                            money  noon  now  offer  office  please  prize  \
spam: free money win prize      1     0    0     

In [7]:
# ── Train & Predict ────────────────────────────────────────────────────────
clf = MultinomialNB(alpha=1.0)   # alpha=1.0 → Laplace smoothing (matches manual)
clf.fit(X_train, y_train)

# Show learned priors
print("=" * 55)
print("sklearn Learned Log Priors")
print("=" * 55)
for cls, log_p in zip(clf.classes_, clf.class_log_prior_):
    print(f"  log P({cls.upper():5s}) = {log_p:.4f}  →  P = {np.exp(log_p):.4f}")

# Show feature log-likelihoods for test words
print()
print("=" * 55)
print("sklearn Feature Log-Likelihoods (test words only)")
print("=" * 55)
feature_names = vectorizer.get_feature_names_out()
for word in test_words:
    if word in vectorizer.vocabulary_:
        idx = vectorizer.vocabulary_[word]
        for i, cls in enumerate(clf.classes_):
            log_lh = clf.feature_log_prob_[i][idx]
            print(f"  log P('{word}' | {cls.upper():5s}) = {log_lh:.4f}  → P = {np.exp(log_lh):.4f}")
    else:
        print(f"  '{word}' NOT in vocabulary")
    print()

sklearn Learned Log Priors
  log P(HAM  ) = -0.6931  →  P = 0.5000
  log P(SPAM ) = -0.6931  →  P = 0.5000

sklearn Feature Log-Likelihoods (test words only)
  log P('free' | HAM  ) = -3.3322  → P = 0.0357
  log P('free' | SPAM ) = -1.9459  → P = 0.1429

  log P('money' | HAM  ) = -3.3322  → P = 0.0357
  log P('money' | SPAM ) = -2.6391  → P = 0.0714

  log P('today' | HAM  ) = -1.9459  → P = 0.1429
  log P('today' | SPAM ) = -3.3322  → P = 0.0357



In [8]:
# ── Predict test message ───────────────────────────────────────────────────
X_test      = vectorizer.transform([test_message])
pred_label  = clf.predict(X_test)[0]
pred_proba  = clf.predict_proba(X_test)[0]

print("=" * 55)
print(f"sklearn Prediction for: '{test_message}'")
print("=" * 55)
for cls, prob in zip(clf.classes_, pred_proba):
    bar = "█" * int(prob * 40)
    print(f"  {cls.upper():5s}: {prob*100:5.2f}%  {bar}")
print()
print(f"  ✅ SKLEARN PREDICTION: {pred_label.upper()}")

sklearn Prediction for: 'free money today'
  HAM  : 33.33%  █████████████
  SPAM : 66.67%  ██████████████████████████

  ✅ SKLEARN PREDICTION: SPAM


In [9]:
# ─── Extended Dataset — Test Multiple Messages ──────────────────────────────
test_messages = [
    "free money today",
    "call me at office",
    "win free prize now",
    "lunch meeting today",
    "lottery offer free",
]

X_multi = vectorizer.transform(test_messages)
preds   = clf.predict(X_multi)
probas  = clf.predict_proba(X_multi)

print("=" * 65)
print("Predictions on Multiple Test Messages")
print("=" * 65)
print(f"  {'Message':<30}  {'Prediction':>10}  {'P(HAM)':>8}  {'P(SPAM)':>8}")
print("  " + "-"*58)
for msg, pred, prob in zip(test_messages, preds, probas):
    idx_ham  = list(clf.classes_).index('ham')
    idx_spam = list(clf.classes_).index('spam')
    print(f"  {msg:<30}  {pred.upper():>10}  {prob[idx_ham]*100:>7.2f}%  {prob[idx_spam]*100:>7.2f}%")

Predictions on Multiple Test Messages
  Message                         Prediction    P(HAM)   P(SPAM)
  ----------------------------------------------------------
  free money today                      SPAM    33.33%    66.67%
  call me at office                      HAM    96.00%     4.00%
  win free prize now                    SPAM     1.03%    98.97%
  lunch meeting today                    HAM    94.12%     5.88%
  lottery offer free                    SPAM     5.88%    94.12%


In [10]:
# ─── Comparison: Manual vs sklearn ─────────────────────────────────────────
total_score    = sum(scores.values())
manual_probs   = {cls: scores[cls] / total_score for cls in classes}

idx_ham  = list(clf.classes_).index('ham')
idx_spam = list(clf.classes_).index('spam')
X_test   = vectorizer.transform([test_message])
sk_proba = clf.predict_proba(X_test)[0]

print("=" * 55)
print(f"COMPARISON for: '{test_message}'")
print("=" * 55)
print(f"  {'Class':<8}  {'Manual':>10}  {'sklearn':>10}  {'Match?':>8}")
print("  " + "-"*45)
for cls, sk_p in [("ham", sk_proba[idx_ham]), ("spam", sk_proba[idx_spam])]:
    man_p = manual_probs.get(cls, 0)
    match = "✅ YES" if abs(man_p - sk_p) < 0.01 else "⚠️ DIFF"
    print(f"  {cls.upper():<8}  {man_p*100:>9.2f}%  {sk_p*100:>9.2f}%  {match:>8}")

print()
print(f"  Manual prediction : {max(scores,       key=scores.get).upper()}")
print(f"  sklearn prediction: {clf.predict(X_test)[0].upper()}")

agree = max(scores, key=scores.get) == clf.predict(X_test)[0]
print()
print(f"  {'✅ Both agree!' if agree else '⚠️ Predictions differ — check smoothing.'}")

COMPARISON for: 'free money today'
  Class         Manual     sklearn    Match?
  ---------------------------------------------
  HAM           33.33%      33.33%     ✅ YES
  SPAM          66.67%      66.67%     ✅ YES

  Manual prediction : SPAM
  sklearn prediction: SPAM

  ✅ Both agree!


---
## Summary

| Concept | Description |
|---|---|
| **Prior P(C)** | Frequency of each class in training data |
| **Likelihood P(w\|C)** | Word frequency per class (with Laplace smoothing) |
| **Posterior P(C\|w)** | Prior × product of likelihoods |
| **Naive Assumption** | All words are independent given the class |
| **Laplace Smoothing** | Add 1 to all counts to avoid zero probabilities |
| **Log Probabilities** | Used in code to avoid numerical underflow |

Both the **manual calculation** and **sklearn** use the same mathematical formula and produce identical (or near-identical) results, confirming correctness.
